In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_collateral(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "CollateralId string, SourceSystem string, PortfolioKey string,"
        # "PortfolioName string, IngestionTimestamp timestamp, BatchId string,"
        # "CollateralKey string, CollateralOldKey string, CollateralCode string, Comment string,"
        # "CollateralAmount decimal(19,2), CollateralCurrency string, CollateralStatusCode string,"
        # "CollateralStatus string,CollateralStatusChangeDate date,CollateralREStatusChangeDate date, StatusDate date, MortgageType string,# "MortgageTypeCode string,"
        # "AgreementKey string,"
        # "CategoryCode string, Category string, SubCategoryCode string, SubCategory string, OwnerPartyKey int, RealEstateKey int,"
        # " NonRealEstateKey int, NonRealEstateCollateralStatusCode string, NonRealEstateCollateralStatus string, OwnershipKey int,"
        # "RealTitleDescription string, OwnershipPercentage double,"
        # "OwnershipStatusCode string, OwnershipStatus string,"
        # "OwnershipStatusDate date, NotaryNumber string, NotaryDate date,"
        # "KtisisWayKey int,RealEstateCollateralKey string, RealEstateCollateralStatusCode string, RealEstateCollateralStatus string,"
        # "InitialLegalOrder int, LawyerPartyKey string,"
        # "JudgementNumber string, JudgementDate date, AppliedDate date,"
        # "RegistryNumber string, LandRegistry int,"
        # "TranscriptNumber string, TranscriptVolume string,"
        # "TranscriptSheet string, CreditorPartyKey string, CourtNumber int,"
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Collateral"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "thesis"
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== prcollateral ==================

        prcollateral_extra_filters = None

        if prcollateral_extra_filters is not None:
            prcollateral_extra_filters.append(date_filter)
            prcollateral_final_filters = " AND ".join(prcollateral_extra_filters)
        else:
            prcollateral_final_filters = date_filter

        prcollateral_selected_fields = [
            "id",
            "old_collateral_id",
            "comments",
            "collateral_amount",
            "currency_code",
            "collateral_status",
            "date_status_change",
            "mortgage_type",
            "cat_code",
            "subcat_code",
            "subcat_descr",
            "portfolio_owner",
            "businessdate",
        ]

        prcollateral = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prcollateral",
            prcollateral_final_filters,
            prcollateral_selected_fields,
        )
        # ================== prcollateraldet ==================

        prcollateraldet_extra_filters = None
        if prcollateraldet_extra_filters is not None:
            prcollateraldet_extra_filters.append(date_filter)
            prcollateraldet_final_filters = " AND ".join(prcollateraldet_extra_filters)
        else:
            prcollateraldet_final_filters = date_filter
        prcollateraldet_selected_fields = [
            "collateral_id",
            "descr",
            "date_applied",
            "id",
            "collateral_status",
        ]
        prcollateraldet = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prcollateraldet",
            prcollateraldet_final_filters,
            prcollateraldet_selected_fields,
        )

        # ================== pragreement_collateral ==================

        pragreement_collateral_extra_filters = None

        if pragreement_collateral_extra_filters is not None:
            pragreement_collateral_extra_filters.append(date_filter)
            pragreement_collateral_final_filters = " AND ".join(
                pragreement_collateral_extra_filters
            )
        else:
            pragreement_collateral_final_filters = date_filter

        pragreement_collateral_selected_fields = [
            "collateral_code",
            "collateral_id",
            "agreement_id",
        ]

        pragreement_collateral = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_pragreement_collateral",
            pragreement_collateral_final_filters,
            pragreement_collateral_selected_fields,
        )

        pragreement_collateral = pragreement_collateral.distinct()

        # ========================== pragreement ==========================

        pragreement_extra_filters = None

        if pragreement_extra_filters is not None:
            pragreement_extra_filters.append(date_filter)
            pragreement_final_filters = " AND ".join(pragreement_extra_filters)
        else:
            pragreement_final_filters = date_filter

        pragreement_selected_fields = ["org_contract_id", "id"]

        pragreement = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_pragreement",
            pragreement_final_filters,
            pragreement_selected_fields,
        )
        # ========================== prcollateral_re ==========================

        prcollateral_re_extra_filters = None

        if prcollateral_re_extra_filters is not None:
            prcollateral_re_extra_filters.append(date_filter)
            prcollateral_re_final_filters = " AND ".join(prcollateral_re_extra_filters)
        else:
            prcollateral_re_final_filters = date_filter

        prcollateral_re_selected_fields = [
            "id",
            "person_id",
            "property_id",
            "initial_order",
            "lawyer_id",
            "judgement_no",
            "date_judgement",
            "registry_no",
            "land_registry_id",
            "transcript_no",
            "transcript_volume",
            "transcript_sheet",
            "creditor_id",
            "court_id",
            "ownership_id",
            "collateral_id",
            "reference_key",
            "collateral_status",
            "date_applied",
            "date_status_change",
        ]

        prcollateral_re = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prcollateral_re",
            prcollateral_re_final_filters,
            prcollateral_re_selected_fields,
        )

        # ========================== prproperty_owner ==========================

        prproperty_owner_extra_filters = None

        if prproperty_owner_extra_filters is not None:
            prproperty_owner_extra_filters.append(date_filter)
            prproperty_owner_final_filters = " AND ".join(
                prproperty_owner_extra_filters
            )
        else:
            prproperty_owner_final_filters = date_filter

        prproperty_owner_selected_fields = [
            "realtitle_descr",
            "percentage",
            "ownership_status",
            "status_date",
            "notary_no",
            "notary_date",
            "ktisisway_id",
            "id",
        ]

        prproperty_owner = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prproperty_owner",
            prproperty_owner_final_filters,
            prproperty_owner_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_thesis",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ========================== Joins ==========================

        prcollateral_pragreement_collateral_join_condition = (
            prcollateral["id"] == pragreement_collateral["collateral_id"]
        )
        pragreement_collateral_pragreement_join_condition = (
            pragreement_collateral["agreement_id"] == pragreement["id"]
        )
        prcollateral_prcollateral_re_join_condition = (
            prcollateral["id"] == prcollateral_re["collateral_id"]
        )
        prcollateral_re_prproperty_owner_join_condition = (
            prcollateral_re["ownership_id"] == prproperty_owner["id"]
        )
        prcollateral_prcollateraldet_join_condition = (
            prcollateraldet["collateral_id"] == prcollateral["id"]
        )
        prcollateral_portfolio_join_condition = (
            portfolio["portfolio_id_thesis"] == prcollateral["portfolio_owner"]
        )

        target_table = (
            prcollateral.join(
                pragreement_collateral,
                prcollateral_pragreement_collateral_join_condition,
                "left",
            )
            .join(
                pragreement, pragreement_collateral_pragreement_join_condition, "left"
            )
            .join(prcollateral_re, prcollateral_prcollateral_re_join_condition, "left")
            .join(
                prproperty_owner,
                prcollateral_re_prproperty_owner_join_condition,
                "left",
            )
            .join(
                prcollateraldet,
                prcollateral_prcollateraldet_join_condition,
                "left",
            )
            .join(
                portfolio,
                prcollateral_portfolio_join_condition,
                "left",
            )
        )

        primary_key_fields = [
            prcollateral.id,
            pragreement.org_contract_id,
            prcollateral_re.id,
            prcollateraldet.id,
            lit("thesis"),
        ]

        target_table = target_table.withColumn(
            "CollateralId", sha2(concat_ws(";", *primary_key_fields), 256)
        ).withColumn(
            "DateApplied",
            when(
                prcollateral_re.collateral_id.isNotNull(), prcollateral_re.date_applied
            )
            .when(
                prcollateraldet.collateral_id.isNotNull(), prcollateraldet.date_applied
            )
            .otherwise(lit(None)),
        )

        result_df = target_table.select(
            target_table.CollateralId,
            lit("thesis").alias("SourceSystem"),
            prcollateral.portfolio_owner.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            prcollateral.id.cast("string").alias("CollateralKey"),
            prcollateral.old_collateral_id.cast("string").alias("CollateralOldKey"),
            pragreement_collateral.collateral_code.alias("CollateralCode"),
            prcollateral.comments.alias("Comment"),
            prcollateral.collateral_amount.alias("CollateralAmount"),
            prcollateral.currency_code.alias("CollateralCurrency"),
            prcollateral.collateral_status.alias("CollateralStatusCode"),
            prcollateral.collateral_status.alias("CollateralStatus"),
            prcollateral.date_status_change.alias("CollateralStatusChangeDate"),
            prcollateral_re.date_status_change.alias("CollateralREStatusChangeDate"),
            prcollateral.date_status_change.alias("StatusDate"),
            prcollateral.mortgage_type.alias("MortgageTypeCode"),
            prcollateral.mortgage_type.alias("MortgageType"),
            pragreement.org_contract_id.alias("AgreementKey"),
            prcollateral.cat_code.alias("CategoryCode"),
            prcollateral.cat_code.alias("Category"),
            prcollateral.subcat_code.alias("SubCategoryCode"),
            prcollateral.subcat_descr.alias("SubCategory"),
            prcollateral_re.person_id.cast("int").alias("OwnerPartyKey"),
            prcollateral_re.property_id.cast("int").alias("RealEstateKey"),
            prcollateraldet.id.cast("int").alias("NonRealEstateKey"),
            prcollateraldet.collateral_status.cast("string").alias("NonRealEstateCollateralStatusCode"),
            prcollateraldet.collateral_status.cast("string").alias("NonRealEstateCollateralStatus"),
            prproperty_owner.id.alias("OwnershipKey"),
            prproperty_owner.realtitle_descr.alias("RealTitleDescription"),
            prproperty_owner.percentage.alias("OwnershipPercentage"),
            prproperty_owner.ownership_status.alias("OwnershipStatusCode"),
            prproperty_owner.ownership_status.alias("OwnershipStatus"),
            prproperty_owner.status_date.alias("OwnershipStatusDate"),
            prproperty_owner.notary_no.alias("NotaryNumber"),
            prproperty_owner.notary_date.alias("NotaryDate"),
            prproperty_owner.ktisisway_id.cast("int").alias("KtisisWayKey"),
            prcollateral_re.id.cast("string").alias("RealEstateCollateralKey"),
            prcollateral_re.collateral_status.cast("string").alias(
                "RealEstateCollateralStatusCode"
            ),
            prcollateral_re.collateral_status.cast("string").alias(
                "RealEstateCollateralStatus"
            ),
            prcollateral_re.initial_order.alias("InitialLegalOrder"),
            prcollateral_re.lawyer_id.cast("string").alias("LawyerPartyKey"),
            prcollateral_re.judgement_no.alias("JudgementNumber"),
            prcollateral_re.date_judgement.alias("JudgementDate"),
            target_table.DateApplied.alias("AppliedDate"),
            prcollateral_re.registry_no.alias("RegistryNumber"),
            prcollateral_re.land_registry_id.alias("LandRegistry"),
            prcollateral_re.transcript_no.alias("TranscriptNumber"),
            prcollateral_re.transcript_volume.alias("TranscriptVolume"),
            prcollateral_re.transcript_sheet.alias("TranscriptSheet"),
            prcollateral_re.creditor_id.cast("string").alias("CreditorPartyKey"),
            prcollateral_re.court_id.alias("CourtNumber"),
            prcollateral.businessdate.alias("BusinessDate"),
        )

        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_collateral(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)